# 5. AIND Ephys Curation

Build an `AINDEPhysCurationScanConfig`, expand it with `GridScanGenerationTask`, and run `AINDEPhysCurationTask` for each coordinate. For every postprocessed recording the task labels each unit three ways:

1. the default QC, pass or fail on thresholds of refractory-period contamination, presence ratio and amplitude cutoff;
2. UnitRefine, machine-learning classifiers of noise, single and multi units, downloaded from Hugging Face;
3. Bombcell, metric thresholds labelling units as good, MUA, noise or non-somatic.

The noise strategy decides which labels flag a unit as noise (by default UnitRefine and Bombcell must agree). SLAy then proposes merges among the remaining units. The labels go to `unit_labels_<name>.csv`, the merges to `unit_merges_<name>.json`, and both to a spikeinterface curation in `curation_<name>.json`.

On a 9 s recording the default QC fails every unit: its thresholds assume long recordings.

## Imports

In [1]:
import json
from pathlib import Path

import pandas as pd

from obi_one.core.run_tasks import run_tasks_for_generated_scan
from obi_one.core.scan_generation import GridScanGenerationTask
from obi_one.scientific.tasks.aind_ephys._05_curation.config import AINDEPhysCurationScanConfig
from obi_one.scientific.tasks.aind_ephys.blocks import JobKwargs

## Build the scan config

In [2]:
obi_output = (Path.cwd() / "../../../../../../obi-output").resolve()
postprocessing_output_path = obi_output / "04_aind_ephys_postprocessing" / "grid_scan" / "0"
assert postprocessing_output_path.exists(), "Run notebook 04 first."

scan_config = AINDEPhysCurationScanConfig(
    initialize=AINDEPhysCurationScanConfig.Initialize(
        postprocessing_output_path=postprocessing_output_path,
    ),
    job_kwargs=JobKwargs(n_jobs=1),
)

## Generate the grid scan and run the curation task

In [3]:
grid_scan = GridScanGenerationTask(
    form=scan_config,
    output_root=obi_output / "05_aind_ephys_curation" / "grid_scan",
    coordinate_directory_option="ZERO_INDEX",
)
grid_scan.execute()

run_tasks_for_generated_scan(grid_scan)

/Users/james/Documents/obi/code/obi-main/worktree-groups/spike_sorting_pipeline/obi-one/.venv/lib/python3.12/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.4.2 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Users/james/Documents/obi/code/obi-main/worktree-groups/spike_sorting_pipeline/obi-one/.venv/lib/python3.12/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.4.2 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Users/james/Documents/obi/code/obi-main/worktree

/Users/james/Documents/obi/code/obi-main/worktree-groups/spike_sorting_pipeline/obi-one/.venv/lib/python3.12/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator RobustScaler from version 1.4.2 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


[PosixPath('/Users/james/Documents/obi/code/obi-main/worktree-groups/spike_sorting_pipeline/obi-output/05_aind_ephys_curation/grid_scan/0')]

## Inspect the labels, the noise units and the proposed merges

In [4]:
coord_dir = Path(grid_scan.single_configs[0].coordinate_output_root)
for labels_file in sorted(coord_dir.glob("unit_labels_*.csv")):
    name = labels_file.stem.removeprefix("unit_labels_")
    curation = json.loads((coord_dir / f"curation_{name}.json").read_text())
    print(name)
    print("  noise units removed:", curation["removed"])
    print("  proposed merges:", [merge["unit_ids"] for merge in curation["merges"]])
    labels = pd.read_csv(labels_file)
labels

block0_acquisition-ElectricalSeriesRaw_recording1
  noise units removed: [3]
  proposed merges: []


,unit_id,default_qc,unitrefine_label,unitrefine_probability,bombcell_label
0,0,False,mua,0.605632,mua
1,1,False,sua,0.587497,mua
2,3,False,noise,0.787032,noise
3,4,False,mua,0.604812,mua
4,5,False,sua,0.500052,mua
5,6,False,mua,0.526932,mua
6,7,False,sua,0.601076,mua
